<a href="https://colab.research.google.com/github/RushiKP14/Tensorflow/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
#Check whether there is any unknown values.
dataset.isna().sum()

In [ ]:
CATEGORICAL_COLUMNS = ['sex', 'children', 'smoker', 'region']
for feature_name in CATEGORICAL_COLUMNS:
  dataset = pd.get_dummies(dataset, columns=[feature_name], prefix='', prefix_sep='', dtype=float)
dataset.tail()

In [ ]:
#Splitting dataset into training and testing set

#print(len(dataset))
train_dataset = dataset.sample(frac = 0.8)
test_dataset = dataset.drop(train_dataset.index)
#print(train_dataset.shape)
#print(len(test_dataset))
dftrain = train_dataset.reset_index(drop=True)
dftest = test_dataset.reset_index(drop=True)
y_train = dftrain.pop('expenses')
y_test = dftest.pop('expenses')
dftrain.head()


In [ ]:
train_dataset.describe().transpose()[['mean', 'std']]

In [ ]:
#Normalization layer
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(np.array(dftrain))
print(normalizer.mean.numpy())

In [ ]:
#Testing the Normaliation layer
first = np.array(dftrain[:1])

with np.printoptions(precision=2, suppress=True):
  print('First example:', first)
  print()
  print('Normalized:', normalizer(first).numpy())

In [ ]:
#Linear regression

model = tf.keras.Sequential([
    normalizer,
    layers.Dense(units=1)
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='mean_absolute_error')

In [ ]:
history = model.fit(
    dftrain,
    y_train,
    epochs=100)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(dftest, y_test, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
